# FRM Natation — visualisation

In [23]:
import json
from pathlib import Path
import pandas as pd
from setup_env import ensure_project_root
PROJECT_DIR = ensure_project_root()
FRMNATATION_RAW_DIR = PROJECT_DIR / "data" / "raw" / "frmnatation"
HTML_RESULTS_DIR = FRMNATATION_RAW_DIR / "html_results"
PDF_JSON_DIR = FRMNATATION_RAW_DIR / "json_from_pdfs_llamaextract"

## Chargement des données

In [24]:
def _is_nonempty_result_row(row: dict) -> bool:
    """Ignore les lignes vides ou sans temps."""
    temps = (row.get("Temps") or "").strip()
    nom = (row.get("Nom et prénom") or "").strip()
    return bool(temps and nom)


def _tables_to_rows(
    tables: list[dict],
    *,
    source_type: str,
    source_file: str,
    meet_label: str | None = None,
    meet_url: str | None = None,
) -> list[dict]:
    rows: list[dict] = []
    for table in tables:
        category = (table.get("category") or "").strip()
        for row in table.get("rows") or []:
            if not isinstance(row, dict) or not _is_nonempty_result_row(row):
                continue
            record = dict(row)
            record["source_type"] = source_type
            record["source_file"] = source_file
            record["meet_label"] = meet_label or source_file
            record["meet_url"] = meet_url
            record["category"] = category
            rows.append(record)
    return rows


def load_html_results(html_dir: Path = HTML_RESULTS_DIR) -> pd.DataFrame:
    """Charge tous les JSON sous html_results/."""
    all_rows: list[dict] = []
    for path in sorted(html_dir.glob("*.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        all_rows.extend(
            _tables_to_rows(
                payload.get("tables") or [],
                source_type="html",
                source_file=path.name,
                meet_label=payload.get("label"),
                meet_url=payload.get("url"),
            )
        )
    return pd.DataFrame(all_rows)


def load_pdf_json(pdf_dir: Path = PDF_JSON_DIR) -> pd.DataFrame:
    """Charge tous les JSON sous json_from_pdfs_llamaextract/."""
    all_rows: list[dict] = []
    for path in sorted(pdf_dir.glob("*.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        source_file = payload.get("source_file") or path.name
        all_rows.extend(
            _tables_to_rows(
                payload.get("tables") or [],
                source_type="pdf",
                source_file=source_file,
                meet_label=Path(source_file).stem,
            )
        )
    return pd.DataFrame(all_rows)


def load_frmnatation_all() -> pd.DataFrame:
    """Charge et concatène html_results + json_from_pdfs_llamaextract."""
    frames = []
    if HTML_RESULTS_DIR.is_dir():
        frames.append(load_html_results())
    if PDF_JSON_DIR.is_dir():
        frames.append(load_pdf_json())
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

In [18]:
print("Racine projet :", PROJECT_DIR)
print("Dossier FRM Natation :", FRMNATATION_RAW_DIR)
print("  html_results existe :", HTML_RESULTS_DIR.is_dir())
print("  json PDF existe      :", PDF_JSON_DIR.is_dir())

if HTML_RESULTS_DIR.is_dir():
    print("  fichiers HTML JSON  :", len(list(HTML_RESULTS_DIR.glob("*.json"))))
if PDF_JSON_DIR.is_dir():
    print("  fichiers PDF JSON   :", len(list(PDF_JSON_DIR.glob("*.json"))))

df_html = load_html_results() if HTML_RESULTS_DIR.is_dir() else pd.DataFrame()
df_pdf = load_pdf_json() if PDF_JSON_DIR.is_dir() else pd.DataFrame()
df = load_frmnatation_all()

print("\n--- Aperçu ---")
print("Lignes result :", len(df_html))
print("Lignes PDF  :", len(df_pdf))
print("Lignes total:", len(df))


Racine projet : /Users/nouhailaimaneabbassi/Desktop/Pacing
Dossier FRM Natation : /Users/nouhailaimaneabbassi/Desktop/Pacing/data/raw/frmnatation
  html_results existe : True
  json PDF existe      : True
  fichiers HTML JSON  : 43
  fichiers PDF JSON   : 142

--- Aperçu ---
Lignes result : 23343
Lignes PDF  : 27871
Lignes total: 51214


## Colonnes du DataFrame

In [19]:
print(f"Nombre de colonnes : {len(df.columns)}\n")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

Nombre de colonnes : 13

 1. Place
 2. Nom et prénom
 3. Nation
 4. Naissance
 5. Club
 6. Temps
 7. Points
 8. Temps de passage
 9. source_type
10. source_file
11. meet_label
12. meet_url
13. category


## Vérification de la qualité des données

Contrôles sur `Temps` (non vide, convertible en secondes), `Nom et prénom`, `Points`, etc.
Les statuts de course (DSQ, abandon, forfait…) sont identifiés séparément des temps invalides.

In [25]:
import re
import math

from app.scripts.extranat_preprocessing import parse_swim_time_to_seconds

# Codes / libellés connus qui ne sont pas des temps chronométriques
TEMPS_STATUS_PATTERNS = (
    r"^(dns|dnf|dsq|np|n/a|abandon)$",
    r"disqual",
    r"^dsq\b",
    r"^frf\b",
    r"\bnrn\b",
    r"^\d+h\s",  # ex. 1h 11' 30" 62
    r"^\d+'\s",  # ex. 53' 42" 41
)


def _is_temps_status(raw: str) -> bool:
    s = str(raw).strip().lower()
    if not s:
        return False
    return any(re.search(p, s) for p in TEMPS_STATUS_PATTERNS)


def parse_temps_to_seconds(raw) -> float | None:
    """Convertit Temps en secondes. None si vide, statut ou format inconnu."""
    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return None
    s = str(raw).strip().replace(",", ".")
    if not s:
        return None
    if _is_temps_status(s):
        return None

    # ex. "22.91 NRN" → 22.91
    m = re.match(r"^(\d+(?:\.\d+)?)\s+\S+", s)
    if m:
        return float(m.group(1))

    sec = parse_swim_time_to_seconds(s)
    if isinstance(sec, float) and sec == sec:  # pas NaN
        return sec
    return None


def validate_frmnatation_df(data: pd.DataFrame) -> pd.DataFrame:
    """Ajoute des colonnes de contrôle et retourne un résumé."""
    out = data.copy()

    out["temps_raw"] = out["Temps"].astype(str).str.strip()
    out["temps_is_empty"] = out["temps_raw"].eq("") | out["Temps"].isna()
    out["temps_is_status"] = out["temps_raw"].map(_is_temps_status)
    out["Temps_seconds"] = out["Temps"].map(parse_temps_to_seconds)
    out["temps_is_numeric"] = out["Temps_seconds"].notna()
    out["temps_is_invalid"] = (
        ~out["temps_is_empty"] & ~out["temps_is_status"] & ~out["temps_is_numeric"]
    )

    out["nom_is_empty"] = out["Nom et prénom"].isna() | (
        out["Nom et prénom"].astype(str).str.strip() == ""
    )
    out["Points_numeric"] = pd.to_numeric(
        out["Points"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    out["points_is_empty"] = out["Points"].isna() | (out["Points"].astype(str).str.strip() == "")
    out["points_is_numeric"] = out["Points_numeric"].notna()

    out["Naissance_year"] = pd.to_numeric(out["Naissance"], errors="coerce")

    return out


df_checked = validate_frmnatation_df(df)

n = len(df_checked)
checks = {
    "Lignes totales": n,
    "Temps vide / null": int(df_checked["temps_is_empty"].sum()),
    "Temps = statut (DSQ, abandon, forfait…)": int(df_checked["temps_is_status"].sum()),
    "Temps numérique (secondes)": int(df_checked["temps_is_numeric"].sum()),
    "Temps non reconnu (à corriger)": int(df_checked["temps_is_invalid"].sum()),
    "Nom et prénom vide": int(df_checked["nom_is_empty"].sum()),
    "Points vide": int(df_checked["points_is_empty"].sum()),
    "Points non numérique (hors vide)": int(
        (~df_checked["points_is_empty"] & ~df_checked["points_is_numeric"]).sum()
    ),
}

print("--- Résumé des contrôles ---")
for label, count in checks.items():
    pct = 100 * count / n if n else 0
    print(f"  {label}: {count:,} ({pct:.2f} %)")

--- Résumé des contrôles ---
  Lignes totales: 51,214 (100.00 %)
  Temps vide / null: 0 (0.00 %)
  Temps = statut (DSQ, abandon, forfait…): 6,478 (12.65 %)
  Temps numérique (secondes): 44,681 (87.24 %)
  Temps non reconnu (à corriger): 55 (0.11 %)
  Nom et prénom vide: 0 (0.00 %)
  Points vide: 2,398 (4.68 %)
  Points non numérique (hors vide): 100 (0.20 %)


In [26]:
all_numeric = df_checked["temps_is_numeric"].all()
print(f"Toutes les lignes ont un Temps numérique : {all_numeric}")
print(f"Lignes avec Temps_seconds valide : {df_checked['temps_is_numeric'].sum():,} / {len(df_checked):,}")

if df_checked["temps_is_numeric"].any():
    valid = df_checked.loc[df_checked["temps_is_numeric"], "Temps_seconds"]
    print(f"Temps_seconds min : {valid.min():.2f} s")
    print(f"Temps_seconds max : {valid.max():.2f} s")

Toutes les lignes ont un Temps numérique : False
Lignes avec Temps_seconds valide : 44,681 / 51,214
Temps_seconds min : 17.10 s
Temps_seconds max : 3577.79 s


In [27]:
n_pts = (~df_checked["points_is_empty"]).sum()
pct_pts = (
    100 * df_checked.loc[~df_checked["points_is_empty"], "points_is_numeric"].mean()
    if n_pts
    else 0
)

print("Nom et prénom — toutes les lignes renseignées :", df_checked["nom_is_empty"].eq(False).all())
print(f"Points numériques (lignes avec Points non vide) : {pct_pts:.1f} %")
print(
    "Points non numérique (hors vide) :",
    int((~df_checked["points_is_empty"] & ~df_checked["points_is_numeric"]).sum()),
)

n_nais = int(
    (
        df_checked["Naissance"].notna()
        & (df_checked["Naissance"].astype(str).str.strip() != "")
        & df_checked["Naissance_year"].isna()
    ).sum()
)
print(f"Naissance non numérique (hors vide) : {n_nais}")

Nom et prénom — toutes les lignes renseignées : True
Points numériques (lignes avec Points non vide) : 99.8 %
Points non numérique (hors vide) : 100
Naissance non numérique (hors vide) : 23
